In [1]:
# 1 - CREATE - 2_QEMU_RUN.txt - START

"""
Compile and execute RISC-V assembly test cases individually using QEMU with a standalone bare-metal test harness, excluding unsupported instructions, CSRs, and inactive ISA extensions. Initialize architectural states, handle traps, and collect register values and execution errors through GDB.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pathlib import Path
import subprocess, time, os, signal, socket, textwrap, re, sys, pexpect
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = 'TEXT/1_QEMU_INPUT.txt'
FILE_OUTPUT = 'TEXT/2_QEMU_RUN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_FIND_FREE_PORT():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

def FUNCTION_HARD_KILL(STRING_NAME):
    subprocess.run(['pkill', '-9', STRING_NAME], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def FUNCTION_RUN(LIST_COMMAND):
    if FLAG_SHOW_COMMAND:
        print('[cmd]', ' '.join(LIST_COMMAND))
    PROCESS_RESULT = subprocess.run(LIST_COMMAND, cwd=FOLDER_WORKDIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return PROCESS_RESULT

def FUNCTION_START_BG(LIST_COMMAND):
    if FLAG_SHOW_COMMAND:
        print('[bg ]', ' '.join(LIST_COMMAND))
    return subprocess.Popen(LIST_COMMAND, cwd=FOLDER_WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, preexec_fn=os.setsid)

def FUNCTION_KILL_BG(PROCESS_HANDLE):
    if not PROCESS_HANDLE:
        return
    try:
        PGID = os.getpgid(PROCESS_HANDLE.pid)
    except Exception:
        return
    try:
        os.killpg(PGID, signal.SIGTERM)
        PROCESS_HANDLE.wait(timeout=1)
    except Exception:
        try:
            os.killpg(PGID, signal.SIGKILL)
        except Exception:
            pass

def FUNCTION_PORT_OPEN(VALUE_PORT):
    try:
        with socket.create_connection(('127.0.0.1', VALUE_PORT), timeout=VALUE_INTERRUPT_AFTER):
            return True
    except OSError:
        return False

def FUNCTION_WAIT_PORT(VALUE_PORT):
    for _ in range(VALUE_WAIT_TRY):
        if FUNCTION_PORT_OPEN(VALUE_PORT):
            return True
        time.sleep(VALUE_INTERRUPT_AFTER)
    return False

def FUNCTION_EXPECT_PROMPT(PROCESS_GDB):
    for _ in range(VALUE_WAIT_TRY):
        try:
            PROCESS_GDB.expect_exact('(gdb)', timeout=VALUE_INTERRUPT_AFTER)
            return True
        except pexpect.TIMEOUT:
            continue
    return False

def FUNCTION_SPLIT_TEST_CASE(STRING_TEXT):
    LIST_BLOCK = []
    LIST_CURRENT = []
    for EACH_LINE in STRING_TEXT.splitlines():
        if EACH_LINE.strip() == '':
            if LIST_CURRENT:
                LIST_BLOCK.append('\n'.join(LIST_CURRENT))
                LIST_CURRENT = []
        else:
            LIST_CURRENT.append(EACH_LINE)
    if LIST_CURRENT:
        LIST_BLOCK.append('\n'.join(LIST_CURRENT))
    return LIST_BLOCK

def FUNCTION_TIDY_OUTPUT_FILE(FILE_NAME):
    with open(FILE_NAME, 'r', encoding='utf-8') as FILE_HANDLE:
        LIST_LINE = FILE_HANDLE.readlines()
    VALUE_START = 0
    while VALUE_START < len(LIST_LINE) and not (LIST_LINE[VALUE_START].startswith('============') or 'START ITER' in LIST_LINE[VALUE_START]):
        VALUE_START += 1
    LIST_KEPT = LIST_LINE[VALUE_START:]
    LIST_COLLAPSED = []
    VALUE_BLANK_STREAK = 0
    for EACH_LINE in LIST_KEPT:
        if EACH_LINE.strip() == '':
            VALUE_BLANK_STREAK += 1
            if VALUE_BLANK_STREAK <= 1:
                LIST_COLLAPSED.append(EACH_LINE)
        else:
            VALUE_BLANK_STREAK = 0
            LIST_COLLAPSED.append(EACH_LINE)
    while LIST_COLLAPSED and LIST_COLLAPSED[-1].strip() == '':
        LIST_COLLAPSED.pop()
    if LIST_COLLAPSED and LIST_COLLAPSED[-1].endswith('\n'):
        LIST_COLLAPSED[-1] = LIST_COLLAPSED[-1].rstrip('\n')
    with open(FILE_NAME, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.writelines(LIST_COLLAPSED)

def IS_ECHO_WRAP_LINE(s: str) -> bool:
    s2 = s.strip()
    if not s2:
        return True
    if '0x' not in s2 and re.fullmatch(r'[<\d\sA-Za-z0-9_]+', s2):
        if any(tok in s2.split() for tok in ['priv', 'pc', 'mcause', 'f0', 'ft0', 'v0', 'a0', 't0']):
            return True
    return False
# FUNCTION - FINISH

# VARIABLE - BEGIN
FOLDER_WORKDIR = Path('.').resolve()
VALUE_GDB_PORT = FUNCTION_FIND_FREE_PORT()
VALUE_INTERRUPT_AFTER = 0.2
VALUE_WAIT_TRY = 50
FLAG_SHOW_COMMAND = False

##################################################
##################################################
##################################################
##################################################
##################################################
# IMPORTANT - BEGIN
LIST_NOT_IMPLEMENTED = ['sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6', 'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6', 'jvt', 'mnstatus', 'mncause', 'tcontrol', 'mireg', 'mireg2', 'mireg3', 'mireg4', 'mireg5', 'mireg6', 'mtopei', *[f'hpmcounter{i}' for i in range(19, 32)], *[f'hpmcounter{i}h' for i in range(3, 32)], *[f'mhpmcounter{i}' for i in range(19, 32)], *[f'mhpmcounter{i}h' for i in range(3, 32)], 'pmpcfg1', 'pmpcfg3', 'pmpcfg5', 'pmpcfg7', 'pmpcfg9', 'pmpcfg11', 'pmpcfg13', 'pmpcfg15', 'mctrctl', 'mcyclecfg', 'mcyclecfgh', 'mcycleh', 'medelegh', 'menvcfgh', *[f'mhpmevent{i}h' for i in range(3, 32)], 'midelegh', 'mieh', 'minstretcfg', 'minstretcfgh', 'minstreth', 'mintstatus', 'miph', 'mnepc', 'mnscratch', 'mnxti', 'mscratchcsw', 'mscratchcswl', 'mseccfgh', 'mstatush', 'mtvt', 'mvienh', 'mviph', 'hcontext', 'hedelegh', 'henvcfgh', 'hidelegh', 'htimedeltah', 'hvienh', 'hviph', 'hviprio1h', 'hviprio2h', 'mscontext', 'scontext', 'sctrctl', 'sctrdepth', 'sctrstatus', 'sieh', 'sintstatus', 'siph', 'snxti', 'srmcfg', 'sscratchcsw', 'sscratchcswl', 'stimecmph', 'stopei', 'stvt', 'cycleh', 'instreth', 'timeh', 'uintstatus', 'unxti', 'uscratchcsw', 'uscratchcswl', 'utvt', 'vsieh', 'vsiph', 'vstimecmph', 'vstopei', 'scountovf', 'stimecmp', 'vstimecmp', 'hstateen0h', 'hstateen1h', 'hstateen2h', 'hstateen3h', 'mstateen0h', 'mstateen1h', 'mstateen2h', 'mstateen3h', 'aes32dsi', 'aes32dsmi', 'aes32esi', 'aes32esmi']
LIST_NOT_IMPLEMENTED += ['tdata3', 'tselect', 'hgeie', 'hgeip']
EXTENSION_NOT_ACTIVE = ['Smdbltrp']
EDIT_INPUT = True
STRING_REGISTER_LIST = 'priv mcause a0 a1 mcycle minstret'
IS_OUTPUT = False
IS_ALL = True
if IS_ALL:
    STRING_REGISTER_LIST = 'all'
# IMPORTANT - FINISH
##################################################
##################################################
##################################################
##################################################
##################################################

REGEX_ANSI_ESCAPE = re.compile(r'\x1B\[[0-?]*[ -/]*[@-~]')

STRING_TEST_CASE = (FOLDER_WORKDIR / FILE_INPUT).read_text(encoding='utf-8')

if EDIT_INPUT:
    SET_NOT_IMPLEMENTED = {EACH_ITEM.lower() for EACH_ITEM in LIST_NOT_IMPLEMENTED}
    SET_EXTENSION_NOT_ACTIVE = {EACH_ITEM.lower() for EACH_ITEM in EXTENSION_NOT_ACTIVE}

    def FUNCTION_DOES_EXTENSION_NOT_ACTIVE(STRING_BLOCK: str) -> bool:
        LIST_MATCH = re.findall(
            r'(?im)^#\s*(?:\d+\s*)?(?:#\s*)?Extension\s+Must\s+Be\s+Active\s*:\s*(.+?)\s*$',
            STRING_BLOCK
        )

        for EACH_EXTENSION_TEXT in LIST_MATCH:
            LIST_EXTENSION = re.findall(r'[A-Za-z][A-Za-z0-9_]*', EACH_EXTENSION_TEXT)

            for EACH_EXTENSION in LIST_EXTENSION:
                if EACH_EXTENSION.lower() in SET_EXTENSION_NOT_ACTIVE:
                    return True

        return False

    def FUNCTION_IS_TEST_CASE_VALID(STRING_BLOCK: str) -> bool:
        if FUNCTION_DOES_EXTENSION_NOT_ACTIVE(STRING_BLOCK):
            return False

        LIST_LINE = STRING_BLOCK.replace('\r\n', '\n').splitlines()

        try:
            INDEX_ASSEMBLY = LIST_LINE.index('##### ASSEMBLY #####')
        except ValueError:
            return True

        LIST_ASSEMBLY_LINE = []
        for EACH_LINE in LIST_LINE[INDEX_ASSEMBLY + 1:]:
            STRING_LINE_STRIP = EACH_LINE.strip()
            if not STRING_LINE_STRIP:
                continue
            if STRING_LINE_STRIP.startswith('# Result:'):
                break
            if STRING_LINE_STRIP.startswith('#'):
                continue
            LIST_ASSEMBLY_LINE.append(STRING_LINE_STRIP)

        STRING_ASSEMBLY = '\n'.join(LIST_ASSEMBLY_LINE).lower()

        for EACH_NOT_IMPLEMENTED in SET_NOT_IMPLEMENTED:
            if re.search(
                r'(?<![A-Za-z0-9_])' + re.escape(EACH_NOT_IMPLEMENTED) + r'(?![A-Za-z0-9_])',
                STRING_ASSEMBLY
            ):
                return False

        return True

    LIST_BLOCK_RAW = FUNCTION_SPLIT_TEST_CASE(STRING_TEST_CASE)

    LIST_BLOCK_FILTERED = [
        EACH_BLOCK
        for EACH_BLOCK in LIST_BLOCK_RAW
        if FUNCTION_IS_TEST_CASE_VALID(EACH_BLOCK)
    ]

    STRING_TEST_CASE = '\n\n'.join(LIST_BLOCK_FILTERED)

LIST_TEST_CASE = FUNCTION_SPLIT_TEST_CASE(STRING_TEST_CASE)
# VARIABLE - FINISH

# CODE - BEGIN
if IS_OUTPUT:
    class CLASS_STDOUT_TO_FILE:
        def __init__(self, FILE_NAME):
            self.file = open(FILE_NAME, 'w', encoding='utf-8')
            self.stdout = sys.__stdout__
        def write(self, STRING_MESSAGE):
            self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
            self.stdout.write(STRING_MESSAGE)
        def flush(self):
            self.file.flush()
            self.stdout.flush()
        def close(self):
            self.file.close()
else:
    class CLASS_STDOUT_TO_FILE:
        def __init__(self, FILE_NAME):
            self.file = open(FILE_NAME, 'w', encoding='utf-8')
        def write(self, STRING_MESSAGE):
            self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
        def flush(self):
            self.file.flush()
        def close(self):
            self.file.close()

FILE_STDOUT = CLASS_STDOUT_TO_FILE(FILE_OUTPUT)
STDOUT_OLD = sys.stdout
sys.stdout = FILE_STDOUT

try:
    for EACH_INDEX, EACH_SNIPPET in enumerate(LIST_TEST_CASE, 1):
        print('============')
        print(f'START ITER {EACH_INDEX}\n')
        print(EACH_SNIPPET)
        print('')
        STRING_CODE = f'''.section .data
.balign 16
data:
.zero 4096
data_end:
.section .text
.globl _start
.globl test_start
_start:
test_start:
li t0, (1 << 8) | (1 << 9) # ECALL from U and S
csrrc x0, medeleg, t0
li a0, -1
srli a0, a0, 2
csrw pmpaddr0, a0
li a0, 0x0f
csrw pmpcfg0, a0
la t0, trap_vector
csrw mtvec, t0
csrs mstatus, 1 << 3
csrr t0, mhartid
mv tp, t0
csrr t1, mstatus
li t2, 0x1800
not t3, t2
and t1, t1, t3
li t3, (3 << 11)
or t1, t1, t3
li t2, (1 << 13)
or t1, t1, t2
li t2, (1 << 10)
or t1, t1, t2
csrw mstatus, t1
csrw satp, zero
li t0, -1
srli t0, t0, 2
csrw pmpaddr0, t0
li t0, 0x0f
csrw pmpcfg0, t0
li t0, 0xffff
csrw mideleg, t0
csrr t1, sie
li t0, ((1<<1) | (1<<5) | (1<<9))
or t1, t1, t0
csrw sie, t1
la t0, trap_vector
csrw mtvec, t0
li x0, 0
li x1, 0
li x2, 0
li x3, 0
li x4, 0
li x5, 0
li x6, 0
li x7, 0
li x8, 0
li x9, 0
li x10, 0
li x11, 0
li x12, 0
li x13, 0
li x14, 0
li x15, 0
li x16, 0
li x17, 0
li x18, 0
li x19, 0
li x20, 0
li x21, 0
li x22, 0
li x23, 0
li x24, 0
li x25, 0
li x26, 0
li x27, 0
li x28, 0
li x29, 0
li x30, 0
li x31, 0
fmv.d.x f0, x0
fmv.d.x f1, x0
fmv.d.x f2, x0
fmv.d.x f3, x0
fmv.d.x f4, x0
fmv.d.x f5, x0
fmv.d.x f6, x0
fmv.d.x f7, x0
fmv.d.x f8, x0
fmv.d.x f9, x0
fmv.d.x f10, x0
fmv.d.x f11, x0
fmv.d.x f12, x0
fmv.d.x f13, x0
fmv.d.x f14, x0
fmv.d.x f15, x0
fmv.d.x f16, x0
fmv.d.x f17, x0
fmv.d.x f18, x0
fmv.d.x f19, x0
fmv.d.x f20, x0
fmv.d.x f21, x0
fmv.d.x f22, x0
fmv.d.x f23, x0
fmv.d.x f24, x0
fmv.d.x f25, x0
fmv.d.x f26, x0
fmv.d.x f27, x0
fmv.d.x f28, x0
fmv.d.x f29, x0
fmv.d.x f30, x0
fmv.d.x f31, x0
vsetvli x0, x0, e32, m1, ta, ma
vmv.v.i v0, 0
vmv.v.i v1, 0
vmv.v.i v2, 0
vmv.v.i v3, 0
vmv.v.i v4, 0
vmv.v.i v5, 0
vmv.v.i v6, 0
vmv.v.i v7, 0
vmv.v.i v8, 0
vmv.v.i v9, 0
vmv.v.i v10, 0
vmv.v.i v11, 0
vmv.v.i v12, 0
vmv.v.i v13, 0
vmv.v.i v14, 0
vmv.v.i v15, 0
vmv.v.i v16, 0
vmv.v.i v17, 0
vmv.v.i v18, 0
vmv.v.i v19, 0
vmv.v.i v20, 0
vmv.v.i v21, 0
vmv.v.i v22, 0
vmv.v.i v23, 0
vmv.v.i v24, 0
vmv.v.i v25, 0
vmv.v.i v26, 0
vmv.v.i v27, 0
vmv.v.i v28, 0
vmv.v.i v29, 0
vmv.v.i v30, 0
vmv.v.i v31, 0
csrw mcause, x0

# Make the initial PC the same across emulators for AUIPC
li x0, 0
li x0, 0
li x0, 0
li x0, 0
li x0, 0
c.addi x0, 0

{EACH_SNIPPET}

test_end:
j test_end

j .
.align 2
trap_vector:
csrr t5, mcause
csrr s1, mepc
addi s1, s1, 4
csrw mepc, s1
mret
spin:
j spin'''
        (FOLDER_WORKDIR / 'code.S').write_text(textwrap.dedent(STRING_CODE).lstrip())
        PROCESS_QEMU = None
        PROCESS_GDB = None
        try:
            PROCESS_BUILD = FUNCTION_RUN(['riscv64-unknown-elf-gcc', '-march=rv64imafdcvh_zicfiss_zicbom_zicboz_zicsr_zca_zimop_zcmop_zbb_zbs_zkne_zbkb_zabha_zacas_zawrs_zkr_smepmp_zcb_zicond_zba_zknd_zbc_zbkc_zfh_zfbfmin_zfhmin_zfa_zifencei_zvfbfmin_zbkx_zvksed_zvksh_zvknha_zvknhb_zvkg_zvfbfwma_zvbc_zvbb_zvkned_zksed_zksh_zknh_zvkb_zicbop_zicfilp_svinval_zve32f_zicntr_zihpm_smstateen_smaia_sscofpmf_sstc', '-mabi=lp64', '-mcmodel=medany', '-nostdlib', '-nostartfiles', '-T', 'linker.ld', 'code.S', 'machine_to_supervisor.S', 'machine_to_user.S', 'supervisor_to_user.S', 'machine_to_virtual_user.S', 'machine_to_virtual.S', '-o', 'code.elf'])
            if PROCESS_BUILD.returncode != 0:
                print(PROCESS_BUILD.stderr.strip() or '(no stderr)')
                print(f'END ITER {EACH_INDEX} (BUILD FAILED)')
                print('==========\n')
                continue
            PROCESS_QEMU = FUNCTION_START_BG(['qemu-system-riscv64', '-machine', 'virt', '-m', '256M', '-bios', 'none', '-kernel', 'code.elf', '-serial', 'null', '-display', 'none', '-S', '-gdb', f'tcp::{VALUE_GDB_PORT}', '-accel', 'tcg', '-icount', 'shift=0,align=off,sleep=off', '-cpu', 'rv64,v=on,smstateen=on,sscofpmf=on,smcsrind=on,sscsrind=on,smaia=on,ssaia=on,ssccfg=on,smcdeleg=on,zicfiss=on,zimop=on,zcmop=on,zaamo=on,zca=on,zbb=on,zbs=on,zkne=on,zbkb=on,zabha=on,zacas=on,zawrs=on,zkr=on,smepmp=on,zcb=on,zicond=on,i=on,m=on,a=on,f=on,d=on,c=on,h=on,zicsr=on,zicbom=on,zicboz=on,zba=on,zknd=on,zbc=on,zbkc=on,zfh=on,zfbfmin=on,zfhmin=on,zfa=on,zifencei=on,zvfbfmin=on,zbkx=on,zvksed=on,zvksh=on,zvknha=on,zvknhb=on,zvkg=on,zvfbfwma=on,zvbc=on,zvbb=on,zvkned=on,zksed=on,zksh=on,zknh=on,zvkb=on,zicbop=on,zicfilp=on,svinval=on,zve32f=on,zicntr=on,zihpm=on,smstateen=on,smaia=on,sscofpmf=on,sstc=on'])
            if not FUNCTION_WAIT_PORT(VALUE_GDB_PORT):
                print(f'END ITER {EACH_INDEX} (QEMU GDB PORT {VALUE_GDB_PORT} NOT OPEN)')
                print('==========\n')
                continue
            PROCESS_GDB = pexpect.spawn('riscv64-unknown-elf-gdb code.elf', cwd=str(FOLDER_WORKDIR), encoding='utf-8', timeout=VALUE_INTERRUPT_AFTER)
            if not FUNCTION_EXPECT_PROMPT(PROCESS_GDB):
                raise RuntimeError('gdb prompt not ready')
            PROCESS_GDB.sendline('set pagination off')
            FUNCTION_EXPECT_PROMPT(PROCESS_GDB)
            PROCESS_GDB.sendline(f'target remote :{VALUE_GDB_PORT}')
            if not FUNCTION_EXPECT_PROMPT(PROCESS_GDB):
                raise RuntimeError('gdb prompt not returned after target remote')
            PROCESS_GDB.sendline('c')
            time.sleep(VALUE_INTERRUPT_AFTER)
            PROCESS_GDB.sendcontrol('c')
            if not FUNCTION_EXPECT_PROMPT(PROCESS_GDB):
                print(f'END ITER {EACH_INDEX} (GDB PROMPT NOT RETURNED AFTER INTERRUPT)')
                print('==========\n')
                continue
            PROCESS_GDB.sendline(f'info registers {STRING_REGISTER_LIST}')
            if not FUNCTION_EXPECT_PROMPT(PROCESS_GDB):
                print(f'END ITER {EACH_INDEX} (INFO REGISTERS FAILED)')
                print('==========\n')
                continue
            STRING_RAW = PROCESS_GDB.before
            LIST_LINE = [EACH_LINE.rstrip() for EACH_LINE in STRING_RAW.splitlines()]
            LIST_REGISTER_LINE = []
            for EACH_LINE in LIST_LINE:
                STRING_LINE = EACH_LINE.strip()
                if STRING_LINE.startswith('info registers'):
                    continue
                if STRING_LINE.startswith('(gdb)'):
                    continue
                if IS_ECHO_WRAP_LINE(EACH_LINE):
                    continue
                LIST_REGISTER_LINE.append(EACH_LINE)
            print('\n'.join(LIST_REGISTER_LINE))
            print(f'\nEND ITER {EACH_INDEX}')
            print('==========\n')
            PROCESS_GDB.sendline('quit')
            try:
                IDX = PROCESS_GDB.expect(
                    [r'Quit anyway\? \(y or n\)', pexpect.EOF, pexpect.TIMEOUT],
                    timeout=VALUE_INTERRUPT_AFTER
                )
                if IDX == 0:
                    PROCESS_GDB.sendline('y')
            except Exception:
                pass
        except KeyboardInterrupt:
            print('Interrupted by user')
            raise
        finally:
            try:
                if PROCESS_GDB is not None:
                    PROCESS_GDB.close(force=True)
            except Exception:
                pass
            FUNCTION_KILL_BG(PROCESS_QEMU)
finally:
    sys.stdout = STDOUT_OLD
    FILE_STDOUT.close()
    FUNCTION_TIDY_OUTPUT_FILE(FILE_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 2_QEMU_RUN.txt - END

5 minute(s) 26.93 second(s)

====
DONE
====


In [2]:
# 2 - CREATE - 3_QEMU_CHECK.txt - START

"""
Validate RISC-V assembly test case results from QEMU with the standalone bare-metal test harness by comparing observed register values against expected results, including CSR bit fields, vector elements, masked expressions, and trap conditions. Identify execution errors and behavioral mismatches while excluding nop-only test cases.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import operator, re, ast
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = 'TEXT/2_QEMU_RUN.txt'
FILE_OUTPUT = 'TEXT/3_QEMU_CHECK.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NORMALIZE_MCAUSE_EXCEPTION_CODE(VALUE_EXCEPTION_CODE: int) -> int:
    if 0x8000000 <= VALUE_EXCEPTION_CODE < 0x8001000:
        return VALUE_EXCEPTION_CODE - 0x8000000
    return VALUE_EXCEPTION_CODE

def FUNCTION_SPLIT_AND(STRING_TAIL: str):
    LIST_PARTS = []
    DEPTH_PAREN = 0
    DEPTH_BRACKET = 0
    DEPTH_BRACE = 0
    CURRENT = ''
    IDX = 0

    while IDX < len(STRING_TAIL):
        CHAR = STRING_TAIL[IDX]

        if CHAR == '(':
            DEPTH_PAREN += 1
            CURRENT += CHAR
        elif CHAR == ')':
            if DEPTH_PAREN > 0:
                DEPTH_PAREN -= 1
            CURRENT += CHAR
        elif CHAR == '[':
            DEPTH_BRACKET += 1
            CURRENT += CHAR
        elif CHAR == ']':
            if DEPTH_BRACKET > 0:
                DEPTH_BRACKET -= 1
            CURRENT += CHAR
        elif CHAR == '{':
            DEPTH_BRACE += 1
            CURRENT += CHAR
        elif CHAR == '}':
            if DEPTH_BRACE > 0:
                DEPTH_BRACE -= 1
            CURRENT += CHAR
        elif CHAR == '&' and DEPTH_PAREN == 0 and DEPTH_BRACKET == 0 and DEPTH_BRACE == 0:
            if CURRENT.strip():
                LIST_PARTS.append(CURRENT.strip())
            CURRENT = ''
        elif CHAR in [',', ';'] and DEPTH_PAREN == 0 and DEPTH_BRACKET == 0 and DEPTH_BRACE == 0:
            STRING_REST = STRING_TAIL[IDX + 1:].strip()
            if re.match(r'^(?:\(?\s*[A-Za-z_]\w*(?:\[[0-9:]+\])?(?:\.(?:b|s|w|l|q))?(?:\[[0-9]+\])?\s*\)?|\([A-Za-z_]\w*\s*&\s*(?:0x[0-9a-fA-F]+|\d+)\)|[A-Za-z_]\w+\s*(?:>>|<<)\s*(?:0x[0-9a-fA-F]+|\d+|[A-Za-z_]\w+))\s*(?:==|=|!=|<=|>=|<|>)', STRING_REST):
                if CURRENT.strip():
                    LIST_PARTS.append(CURRENT.strip())
                CURRENT = ''
            else:
                CURRENT += CHAR
        else:
            CURRENT += CHAR

        IDX += 1

    if CURRENT.strip():
        LIST_PARTS.append(CURRENT.strip())

    return LIST_PARTS

def FUNCTION_IS_NOP_ONLY_TEST(STRING_ITERATION: str) -> bool:
    LIST_CODE = []
    FLAG_IN_CODE = False
    for EACH_LINE in STRING_ITERATION.splitlines():
        STRING_STRIPPED = EACH_LINE.strip()
        if not STRING_STRIPPED:
            continue
        if re.match(r'^\s*#\s*(?:\d+\s*#\s*)?Extension Must Be Active:', EACH_LINE, re.IGNORECASE):
            FLAG_IN_CODE = True
            continue
        if re.match(r'^\s*#\s*Result\s*:', EACH_LINE, re.IGNORECASE):
            break
        if FLAG_IN_CODE:
            if STRING_STRIPPED.startswith('#'):
                continue
            LIST_CODE.append(STRING_STRIPPED)
    return LIST_CODE == ['nop']

def FUNCTION_CANONICAL_REGISTER(STRING_NAME: str) -> str:
    STRING_KEY = STRING_NAME.strip()
    MATCH_V = re.fullmatch(r'(v\d+)\[(\d+)\]', STRING_KEY)
    if MATCH_V:
        return f'{MATCH_V.group(1)}.w[{MATCH_V.group(2)}]'
    MATCH_V2 = re.fullmatch(r'(v\d+)\.(b|s|w|l|q)\[(\d+)\]', STRING_KEY)
    if MATCH_V2:
        return f'{MATCH_V2.group(1)}.{MATCH_V2.group(2)}[{MATCH_V2.group(3)}]'
    return MAP_REGISTER_ALIAS.get(STRING_KEY, STRING_KEY)

def FUNCTION_PARSE_VECTOR_LIST(STRING_BODY: str, TOTAL_LEN: int):
    LIST_OUT = []
    TOKENS = [x.strip() for x in STRING_BODY.split(',') if x.strip()]
    for TOK in TOKENS:
        MREP = re.fullmatch(r'(0x[0-9a-fA-F]+)\s*<repeats\s+(\d+)\s+times>', TOK)
        if MREP:
            VAL = int(MREP.group(1), 16)
            CNT = int(MREP.group(2))
            LIST_OUT.extend([VAL] * CNT)
            continue
        MHEX = re.fullmatch(r'0x[0-9a-fA-F]+', TOK)
        if MHEX:
            LIST_OUT.append(int(TOK, 16))
            continue
        continue
    if len(LIST_OUT) < TOTAL_LEN:
        LIST_OUT.extend([0] * (TOTAL_LEN - len(LIST_OUT)))
    return LIST_OUT[:TOTAL_LEN]

def FUNCTION_EVAL_INT_EXPR(STRING_VALUE: str) -> int:
    STRING_EXPR = STRING_VALUE.strip()
    if not STRING_EXPR:
        raise ValueError('empty expression')
    NODE = ast.parse(STRING_EXPR, mode='eval')
    MAP_BIN_OP = {
        ast.LShift: operator.lshift,
        ast.RShift: operator.rshift,
        ast.BitOr: operator.or_,
        ast.BitAnd: operator.and_,
        ast.BitXor: operator.xor,
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.FloorDiv: operator.floordiv,
        ast.Mod: operator.mod,
    }
    MAP_UNARY_OP = {
        ast.UAdd: operator.pos,
        ast.USub: operator.neg,
        ast.Invert: operator.invert,
    }
    def _EVAL(EACH_NODE):
        if isinstance(EACH_NODE, ast.Expression):
            return _EVAL(EACH_NODE.body)
        if isinstance(EACH_NODE, ast.Constant):
            if isinstance(EACH_NODE.value, int):
                return EACH_NODE.value
            raise ValueError('non-integer constant')
        if isinstance(EACH_NODE, ast.UnaryOp) and type(EACH_NODE.op) in MAP_UNARY_OP:
            return MAP_UNARY_OP[type(EACH_NODE.op)](_EVAL(EACH_NODE.operand))
        if isinstance(EACH_NODE, ast.BinOp) and type(EACH_NODE.op) in MAP_BIN_OP:
            return MAP_BIN_OP[type(EACH_NODE.op)](_EVAL(EACH_NODE.left), _EVAL(EACH_NODE.right))
        raise ValueError(f'unsupported expression: {STRING_EXPR}')
    return _EVAL(NODE)

def FUNCTION_STRIP_TRAILING_EXPLANATION(STRING_VALUE: str) -> str:
    S = STRING_VALUE.strip()
    if not S:
        return S
    DEPTH = 0
    for IDX, CHAR in enumerate(S):
        if CHAR == '(':
            PREFIX = S[:IDX].rstrip()
            if PREFIX:
                LAST = PREFIX[-1]
                if LAST not in '+-*/%&|^<>=(':
                    return PREFIX
            DEPTH += 1
        elif CHAR == ')':
            if DEPTH > 0:
                DEPTH -= 1
    return S

def FUNCTION_PARSE_INT(STRING_VALUE: str) -> int:
    STRING_VALUE = FUNCTION_STRIP_TRAILING_EXPLANATION(STRING_VALUE)
    STRING_LOWER = STRING_VALUE.strip().lower()
    if STRING_LOWER.startswith(('+0x', '-0x', '0x')):
        return int(STRING_VALUE, 16)
    try:
        return int(STRING_VALUE, 10)
    except ValueError:
        return FUNCTION_EVAL_INT_EXPR(STRING_VALUE)

def FUNCTION_PARSE_ITERATION(STRING_CONTENT):
    REGEX_ITERATION = re.compile(r'START ITER (\d+)(.*?)END ITER \1', re.DOTALL)
    LIST_ITERATION = REGEX_ITERATION.findall(STRING_CONTENT)
    return LIST_ITERATION

def FUNCTION_HAS_ERROR(STRING_ITERATION: str) -> bool:
    return re.search(r'\bpriv\b', STRING_ITERATION) is None

def FUNCTION_PARSE_BIT_SELECTOR(STRING_REG_KEY: str):
    MATCH = re.fullmatch(r'([A-Za-z_]\w+)\[(\d+)\]', STRING_REG_KEY)
    if not MATCH:
        return None
    return MATCH.group(1), int(MATCH.group(2))

def FUNCTION_PARSE_EXPECTED_RESULT(STRING_ITERATION: str):
    STRING_REGISTER_PATTERN = (
        r'(?:'
        r'\([A-Za-z_]\w*\s*&\s*(?:0x[0-9a-fA-F]+|\d+)\)'
        r'|[A-Za-z_]\w+\s*(?:>>|<<)\s*(?:0x[0-9a-fA-F]+|\d+|[A-Za-z_]\w+)'
        r'|[A-Za-z_]\w+(?:\[\d+(?::\d+)?\])?'
        r'|v\d+(?:\.(?:b|s|w|l|q))?\[\d+\]'
        r')'
    )
    REGEX_RESULT_LINE = re.compile(r'#\s*Result:\s*(.*)')
    REGEX_CONDITION = re.compile(
        rf'\s*({STRING_REGISTER_PATTERN})\s*(==|=|!=|<=|>=|<|>)\s*(.+?)\s*$'
    )
    LIST_OR_GROUPS = []
    for EACH_LINE in STRING_ITERATION.splitlines():
        MATCH_RESULT = REGEX_RESULT_LINE.search(EACH_LINE)
        if not MATCH_RESULT:
            continue
        STRING_TAIL = MATCH_RESULT.group(1)
        LIST_OR_PART = [x.strip() for x in STRING_TAIL.split('|') if x.strip()]
        for EACH_OR in LIST_OR_PART:
            LIST_AND = []
            LIST_AND_PART = FUNCTION_SPLIT_AND(EACH_OR)
            for EACH_PART in LIST_AND_PART:
                MATCH_COND = REGEX_CONDITION.fullmatch(EACH_PART)
                if not MATCH_COND:
                    continue
                STRING_REG, STRING_OP, STRING_RHS_TOKEN = MATCH_COND.groups()
                STRING_REG = STRING_REG.strip()
                if STRING_OP == '=':
                    STRING_OP = '=='
                LIST_AND.append((STRING_REG, STRING_OP, STRING_RHS_TOKEN.strip()))
            if LIST_AND:
                LIST_OR_GROUPS.append(LIST_AND)
    return LIST_OR_GROUPS

def FUNCTION_PARSE_ACTUAL_REGISTER(STRING_ITERATION: str):
    STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
    REGEX_GPR = re.compile(rf'(\w+)\s+0x[0-9a-fA-F]+\s+({STRING_VALUE_PATTERN})')
    REGEX_HEX_ONLY_LINE = re.compile(r'^(\w+)\s+(0x[0-9a-fA-F]+)(?:\s+.*)?$', re.MULTILINE)
    REGEX_FP_RAW = re.compile(
        r'^(f(?:t|s|a)?\d+)\s+[\s\S]*?\(raw\s+0x([0-9a-fA-F]+)\)',
        re.MULTILINE
    )
    REGEX_F_DOUBLE = re.compile(
        r'^(f\d+)\s+\{float\s*=\s*0x[0-9a-fA-F]+,\s*double\s*=\s*0x([0-9a-fA-F]+)\}',
        re.MULTILINE
    )
    REGEX_MCAUSE = re.compile(
        r'^mcause\s+\{Interrupt\s*=\s*(0x[0-9a-fA-F]+|\d+),\s*Exception Code\s*=\s*(0x[0-9a-fA-F]+|\d+)\}',
        re.MULTILINE
    )
    MAP_ACTUAL = {}
    for EACH_REG, EACH_VALUE in REGEX_GPR.findall(STRING_ITERATION):
        try:
            VALUE_PARSED = FUNCTION_PARSE_INT(EACH_VALUE)
        except ValueError:
            continue
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX in REGEX_HEX_ONLY_LINE.findall(STRING_ITERATION):
        if EACH_REG in MAP_ACTUAL:
            continue
        try:
            VALUE_PARSED = int(EACH_HEX, 16)
        except ValueError:
            continue
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX in REGEX_FP_RAW.findall(STRING_ITERATION):
        VALUE_PARSED = int(EACH_HEX, 16)
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX_DOUBLE in REGEX_F_DOUBLE.findall(STRING_ITERATION):
        VALUE_PARSED = int(EACH_HEX_DOUBLE, 16)
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_INTERRUPT, EACH_EXCEPTION_CODE in REGEX_MCAUSE.findall(STRING_ITERATION):
        try:
            VALUE_INTERRUPT = FUNCTION_PARSE_INT(EACH_INTERRUPT)
            VALUE_EXCEPTION_CODE = FUNCTION_PARSE_INT(EACH_EXCEPTION_CODE)
        except ValueError:
            continue
        VALUE_EXCEPTION_CODE = FUNCTION_NORMALIZE_MCAUSE_EXCEPTION_CODE(VALUE_EXCEPTION_CODE)
        VALUE_MCAUSE = (FUNCTION_U64(VALUE_INTERRUPT) << 63) | FUNCTION_U64(VALUE_EXCEPTION_CODE)
        MAP_ACTUAL['mcause'] = VALUE_MCAUSE
    REGEX_VBLOCK = re.compile(r'^(v\d+)\s+(.*?)(?=^v\d+\s+|\Z)', re.MULTILINE | re.DOTALL)
    REGEX_VIEW = re.compile(r'\b([qlwsb])\s*=\s*\{(.*?)\}', re.DOTALL)
    MAP_LEN = {'b': 16, 's': 8, 'w': 4, 'l': 2, 'q': 1}
    MAP_MASK = {'b': (1 << 8) - 1, 's': (1 << 16) - 1, 'w': (1 << 32) - 1, 'l': (1 << 64) - 1, 'q': MASK64}
    for VREG, VBLOCK in REGEX_VBLOCK.findall(STRING_ITERATION):
        VREG = VREG.strip()
        for VIEW, BODY in REGEX_VIEW.findall(VBLOCK):
            VIEW = VIEW.strip()
            if VIEW not in MAP_LEN:
                continue
            LIST_VAL = FUNCTION_PARSE_VECTOR_LIST(BODY, MAP_LEN[VIEW])
            for IDX, VAL in enumerate(LIST_VAL):
                KEY = f'{VREG}.{VIEW}[{IDX}]'
                MAP_ACTUAL[KEY] = VAL & MAP_MASK[VIEW]
    return MAP_ACTUAL

def FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY: str):
    M = re.fullmatch(r'v\d+\.(b|s|w|l|q)\[\d+\]', STRING_REG_KEY)
    if not M:
        return None
    return {'b': 8, 's': 16, 'w': 32, 'l': 64, 'q': 64}[M.group(1)]

def FUNCTION_HAS_RESULT_ONLY_ERROR(STRING_ITERATION: str) -> bool:
    return REGEX_RESULT_ONLY_ERROR.search(STRING_ITERATION) is not None

def FUNCTION_U64(VALUE_X: int) -> int:
    return VALUE_X & MASK64

def FUNCTION_PARSE_MASK_EXPR(STRING_REG_KEY: str):
    # handle (reg & mask) pattern
    MATCH = re.fullmatch(r'\(\s*([A-Za-z_]\w*)\s*&\s*(0x[0-9a-fA-F]+|\d+)\s*\)', STRING_REG_KEY.strip())
    if not MATCH:
        return None
    return MATCH.group(1), FUNCTION_PARSE_INT(MATCH.group(2))

def FUNCTION_PARSE_BIT_RANGE(STRING_REG_KEY: str):
    # handle reg[hi:lo] pattern
    MATCH = re.fullmatch(r'([A-Za-z_]\w+)\[(\d+):(\d+)\]', STRING_REG_KEY.strip())
    if not MATCH:
        return None
    return MATCH.group(1), int(MATCH.group(2)), int(MATCH.group(3))

def FUNCTION_CHECK_ITERATION(LIST_ITERATION):
    STRING_OUTPUT = ''
    for EACH_ITER_NUMBER, EACH_ITER_CONTENT in LIST_ITERATION:
        VALUE_ITER_NUMBER = int(EACH_ITER_NUMBER)
        if FUNCTION_IS_NOP_ONLY_TEST(EACH_ITER_CONTENT):
            continue
        if FUNCTION_HAS_RESULT_ONLY_ERROR(EACH_ITER_CONTENT):
            if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                continue
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
            continue
        if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is error\n'
            continue
        LIST_EXPECTED_GROUP = FUNCTION_PARSE_EXPECTED_RESULT(EACH_ITER_CONTENT)
        MAP_ACTUAL = FUNCTION_PARSE_ACTUAL_REGISTER(EACH_ITER_CONTENT)
        FLAG_MATCH = False
        for EACH_GROUP in LIST_EXPECTED_GROUP:
            GROUP_OK = True
            for EACH_REG, EACH_OP, EACH_RHS_TOKEN in EACH_GROUP:
                STRING_REG_KEY = FUNCTION_CANONICAL_REGISTER(EACH_REG)
                if FUNCTION_IS_PLAIN_VECTOR_REGISTER(STRING_REG_KEY):
                    LIST_ACTUAL_B = FUNCTION_GET_VECTOR_B_LIST(MAP_ACTUAL, STRING_REG_KEY)
                    if LIST_ACTUAL_B is None:
                        GROUP_OK = False
                        break
                    try:
                        VALUE_EXPECTED = FUNCTION_PARSE_INT(EACH_RHS_TOKEN) & 0xFF
                        if EACH_OP == '=':
                            EACH_OP = '=='
                        if EACH_OP == '==':
                            if not all(EACH_VALUE == VALUE_EXPECTED for EACH_VALUE in LIST_ACTUAL_B):
                                GROUP_OK = False
                                break
                        elif EACH_OP == '!=':
                            if not all(EACH_VALUE != VALUE_EXPECTED for EACH_VALUE in LIST_ACTUAL_B):
                                GROUP_OK = False
                                break
                        else:
                            GROUP_OK = False
                            break
                        continue
                    except ValueError:
                        STRING_RHS_KEY = FUNCTION_CANONICAL_REGISTER(EACH_RHS_TOKEN)
                        if not FUNCTION_IS_PLAIN_VECTOR_REGISTER(STRING_RHS_KEY):
                            GROUP_OK = False
                            break
                        LIST_EXPECTED_B = FUNCTION_GET_VECTOR_B_LIST(MAP_ACTUAL, STRING_RHS_KEY)
                        if LIST_EXPECTED_B is None:
                            GROUP_OK = False
                            break
                        if EACH_OP == '=':
                            EACH_OP = '=='
                        if EACH_OP == '==':
                            if LIST_ACTUAL_B != LIST_EXPECTED_B:
                                GROUP_OK = False
                                break
                        elif EACH_OP == '!=':
                            if LIST_ACTUAL_B == LIST_EXPECTED_B:
                                GROUP_OK = False
                                break
                        else:
                            GROUP_OK = False
                            break
                        continue
                VALUE_ACTUAL = FUNCTION_GET_ACTUAL_VALUE(MAP_ACTUAL, STRING_REG_KEY)
                if VALUE_ACTUAL is None:
                    GROUP_OK = False
                    break
                try:
                    VALUE_EXPECTED = FUNCTION_PARSE_INT(EACH_RHS_TOKEN)
                except ValueError:
                    STRING_RHS_KEY = FUNCTION_CANONICAL_REGISTER(EACH_RHS_TOKEN)
                    VALUE_RHS = FUNCTION_GET_ACTUAL_VALUE(MAP_ACTUAL, STRING_RHS_KEY)
                    if VALUE_RHS is None:
                        GROUP_OK = False
                        break
                    VALUE_EXPECTED = VALUE_RHS
                if EACH_OP in ('==', '!='):
                    WIDTH_BITS = None
                    WIDTH_BITS = FUNCTION_HEX_WIDTH_BITS(EACH_RHS_TOKEN)
                    V_BITS = FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY)
                    if V_BITS is not None:
                        WIDTH_BITS = V_BITS
                    if WIDTH_BITS == 8:
                        VALUE_A = VALUE_ACTUAL & ((1 << 8) - 1)
                        VALUE_E = VALUE_EXPECTED & ((1 << 8) - 1)
                    elif WIDTH_BITS == 16:
                        VALUE_A = VALUE_ACTUAL & ((1 << 16) - 1)
                        VALUE_E = VALUE_EXPECTED & ((1 << 16) - 1)
                    elif WIDTH_BITS == 32:
                        VALUE_A = VALUE_ACTUAL & MASK32
                        VALUE_E = VALUE_EXPECTED & MASK32
                    else:
                        VALUE_A = FUNCTION_U64(VALUE_ACTUAL)
                        VALUE_E = FUNCTION_U64(VALUE_EXPECTED)
                    if not MAP_OPERATOR[EACH_OP](VALUE_A, VALUE_E):
                        GROUP_OK = False
                        break
            if GROUP_OK:
                FLAG_MATCH = True
                break
        if not FLAG_MATCH:
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
    STRING_OUTPUT_FINAL = STRING_OUTPUT.strip()
    if STRING_OUTPUT_FINAL:
        print(STRING_OUTPUT_FINAL)
        print()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_OUTPUT_FINAL)

def FUNCTION_IS_FLOAT_REG(STRING_REG_CANON: str) -> bool:
    return re.fullmatch(r'f(?:[0-9]|[12][0-9]|3[01])', STRING_REG_CANON) is not None

def FUNCTION_HEX_WIDTH_BITS(STRING_TOKEN: str):
    S = STRING_TOKEN.strip().lower()
    if S.startswith(('+0x', '-0x')):
        S = S[1:]
    if not S.startswith('0x'):
        return None
    HEXDIG = S[2:]
    if 1 <= len(HEXDIG) <= 8:
        return 32
    if 9 <= len(HEXDIG) <= 16:
        return 64
    return None

def FUNCTION_IS_PLAIN_VECTOR_REGISTER(STRING_REG_KEY: str) -> bool:
    return re.fullmatch(r'v\d+', STRING_REG_KEY) is not None

def FUNCTION_GET_VECTOR_B_LIST(MAP_ACTUAL: dict, STRING_VREG: str):
    LIST_OUT = []
    for IDX in range(16):
        VALUE = MAP_ACTUAL.get(f'{STRING_VREG}.b[{IDX}]')
        if VALUE is None:
            return None
        LIST_OUT.append(VALUE & 0xFF)
    return LIST_OUT

def FUNCTION_GET_ACTUAL_VALUE(MAP_ACTUAL: dict, STRING_REG_KEY: str):
    STRING_REG_KEY_STRIPPED = STRING_REG_KEY.strip()

    # handle (reg & mask) pattern
    TUPLE_MASK = FUNCTION_PARSE_MASK_EXPR(STRING_REG_KEY_STRIPPED)
    if TUPLE_MASK is not None:
        STRING_BASE_REG, VALUE_MASK = TUPLE_MASK
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(STRING_BASE_REG)
        VALUE_BASE = MAP_ACTUAL.get(STRING_CANON)
        if VALUE_BASE is None:
            return None
        return FUNCTION_U64(VALUE_BASE) & VALUE_MASK

    # handle reg >> n or reg << n
    TUPLE_SHIFT = FUNCTION_PARSE_SHIFT_EXPR(STRING_REG_KEY_STRIPPED)
    if TUPLE_SHIFT is not None:
        STRING_BASE_REG, STRING_SHIFT_OP, STRING_SHIFT_TOKEN = TUPLE_SHIFT
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(STRING_BASE_REG)
        VALUE_BASE = MAP_ACTUAL.get(STRING_CANON)
        if VALUE_BASE is None:
            return None
        VALUE_SHIFT = FUNCTION_PARSE_INT(STRING_SHIFT_TOKEN)
        if STRING_SHIFT_OP == '>>':
            return FUNCTION_U64(VALUE_BASE) >> VALUE_SHIFT
        return FUNCTION_U64(VALUE_BASE) << VALUE_SHIFT

    # handle reg[hi:lo] pattern
    TUPLE_RANGE = FUNCTION_PARSE_BIT_RANGE(STRING_REG_KEY_STRIPPED)
    if TUPLE_RANGE is not None:
        STRING_BASE_REG, VALUE_HI, VALUE_LO = TUPLE_RANGE
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(STRING_BASE_REG)
        VALUE_BASE = MAP_ACTUAL.get(STRING_CANON)
        if VALUE_BASE is None:
            return None
        VALUE_MASK = (1 << (VALUE_HI - VALUE_LO + 1)) - 1
        return (FUNCTION_U64(VALUE_BASE) >> VALUE_LO) & VALUE_MASK

    # existing single bit selector
    STRING_REG_KEY = FUNCTION_CANONICAL_REGISTER(STRING_REG_KEY_STRIPPED)
    TUPLE_BIT = FUNCTION_PARSE_BIT_SELECTOR(STRING_REG_KEY)
    if TUPLE_BIT is not None:
        STRING_BASE_REG, VALUE_BIT_INDEX = TUPLE_BIT
        VALUE_BASE = MAP_ACTUAL.get(STRING_BASE_REG)
        if VALUE_BASE is None:
            return None
        return (FUNCTION_U64(VALUE_BASE) >> VALUE_BIT_INDEX) & 1

    VALUE_DIRECT = MAP_ACTUAL.get(STRING_REG_KEY)
    if VALUE_DIRECT is not None:
        return VALUE_DIRECT
    if FUNCTION_IS_PLAIN_VECTOR_REGISTER(STRING_REG_KEY):
        return MAP_ACTUAL.get(f'{STRING_REG_KEY}.q[0]')
    return None

def FUNCTION_PARSE_SHIFT_EXPR(STRING_REG_KEY: str):
    MATCH = re.fullmatch(
        r'([A-Za-z_]\w+)\s*(>>|<<)\s*(.+)',
        STRING_REG_KEY.strip()
    )
    if not MATCH:
        return None
    STRING_BASE_REG = MATCH.group(1)
    STRING_SHIFT_OP = MATCH.group(2)
    STRING_SHIFT_TOKEN = MATCH.group(3).strip()
    return STRING_BASE_REG, STRING_SHIFT_OP, STRING_SHIFT_TOKEN

def FUNCTION_TRUNCATE_RHS_NATURAL_LANGUAGE(STRING_RHS: str) -> str:
    STRING_RHS = STRING_RHS.strip()
    MATCH = re.search(r'\b(?:and|or|xor)\b', STRING_RHS, re.IGNORECASE)
    if MATCH:
        STRING_RHS = STRING_RHS[:MATCH.start()].strip()
    STRING_RHS = STRING_RHS.rstrip('.').strip()
    return STRING_RHS
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_RESULT_ONLY_ERROR = re.compile(r'^\s*#\s*Result\s*:\s*error\s*$', re.IGNORECASE | re.MULTILINE)
MAP_OPERATOR = {'=': operator.eq, '==': operator.eq, '!=': operator.ne, '<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}
MAP_REGISTER_ALIAS = {'x0': 'x0', 'zero': 'x0', 'x1': 'x1', 'ra': 'x1', 'x2': 'x2', 'sp': 'x2', 'x3': 'x3', 'gp': 'x3', 'x4': 'x4', 'tp': 'x4', 'x5': 'x5', 't0': 'x5', 'x6': 'x6', 't1': 'x6', 'x7': 'x7', 't2': 'x7', 'x8': 'x8', 's0': 'x8', 'fp': 'x8', 'x9': 'x9', 's1': 'x9', 'x10': 'x10', 'a0': 'x10', 'x11': 'x11', 'a1': 'x11', 'x12': 'x12', 'a2': 'x12', 'x13': 'x13', 'a3': 'x13', 'x14': 'x14', 'a4': 'x14', 'x15': 'x15', 'a5': 'x15', 'x16': 'x16', 'a6': 'x16', 'x17': 'x17', 'a7': 'x17', 'x18': 'x18', 's2': 'x18', 'x19': 'x19', 's3': 'x19', 'x20': 'x20', 's4': 'x20', 'x21': 'x21', 's5': 'x21', 'x22': 'x22', 's6': 'x22', 'x23': 'x23', 's7': 'x23', 'x24': 'x24', 's8': 'x24', 'x25': 'x25', 's9': 'x25', 'x26': 'x26', 's10': 'x26', 'x27': 'x27', 's11': 'x27', 'x28': 'x28', 't3': 'x28', 'x29': 'x29', 't4': 'x29', 'x30': 'x30', 't5': 'x30', 'x31': 'x31', 't6': 'x31'}
MASK64 = (1 << 64) - 1
MASK32 = (1 << 32) - 1
MAP_REGISTER_ALIAS.update({**{f'f{i}': f'f{i}' for i in range(32)}, 'ft0': 'f0', 'ft1': 'f1', 'ft2': 'f2', 'ft3': 'f3', 'ft4': 'f4', 'ft5': 'f5', 'ft6': 'f6', 'ft7': 'f7', 'ft8': 'f28', 'ft9': 'f29', 'ft10': 'f30', 'ft11': 'f31', 'fs0': 'f8', 'fs1': 'f9', 'fs2': 'f18', 'fs3': 'f19', 'fs4': 'f20', 'fs5': 'f21', 'fs6': 'f22', 'fs7': 'f23', 'fs8': 'f24', 'fs9': 'f25', 'fs10': 'f26', 'fs11': 'f27', 'fa0': 'f10', 'fa1': 'f11', 'fa2': 'f12', 'fa3': 'f13', 'fa4': 'f14', 'fa5': 'f15', 'fa6': 'f16', 'fa7': 'f17'})
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT = FILE_HANDLE.read()

LIST_ITERATION = FUNCTION_PARSE_ITERATION(STRING_CONTENT)
FUNCTION_CHECK_ITERATION(LIST_ITERATION)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 3_QEMU_CHECK.txt - END

START ITER 2 is false
START ITER 4 is false
START ITER 5 is error
START ITER 6 is error
START ITER 7 is error
START ITER 8 is false
START ITER 15 is false
START ITER 16 is false
START ITER 18 is false
START ITER 19 is false
START ITER 20 is false
START ITER 21 is error
START ITER 22 is false
START ITER 23 is false
START ITER 25 is error
START ITER 26 is false
START ITER 30 is false
START ITER 31 is false
START ITER 33 is false
START ITER 38 is false
START ITER 40 is false
START ITER 42 is false
START ITER 43 is false
START ITER 44 is false
START ITER 46 is false
START ITER 48 is false
START ITER 49 is false
START ITER 50 is false
START ITER 51 is false
START ITER 54 is false
START ITER 57 is error
START ITER 65 is false
START ITER 66 is false
START ITER 67 is false
START ITER 68 is false
START ITER 69 is false
START ITER 75 is false
START ITER 76 is false
START ITER 79 is false
START ITER 80 is false
START ITER 81 is false
START ITER 82 is false
START ITER 83 is false
START ITER 85 is 